# Linear Regression 

In [1]:
# Import
import pandas as pd
import numpy as np
import time
import warnings
warnings.filterwarnings("ignore")

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet, ElasticNetCV
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

np.random.seed(42)

In [2]:

# Load train & test 
train_df = pd.read_csv("../data/processed/train_data_final.csv")
test_df = pd.read_csv("../data/processed/test_data_final.csv")

In [3]:
# Split train / target
TARGET = "quantity_sold"
LEAK_COL = "review_to_sold_ratio"

X_train = train_df.drop(columns=[TARGET])
y_train = train_df[TARGET]
# drop cột review_to_sold_ratio vì là leak column
X_train = X_train.drop(columns=[LEAK_COL])

X_test = test_df.drop(columns=[TARGET])
y_test = test_df[TARGET]
X_test = X_test.drop(columns=[LEAK_COL])

In [4]:
# Load Model ElasticNet lần 1 (trong model đã chọn bộ tham số tốt nhất bằng ELasticNetCV)
enet_cv = Pipeline([
    ("scaler", StandardScaler()),
    ("model", ElasticNetCV(
        l1_ratio=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99],
        alphas=[0.0001, 0.001, 0.01, 0.1, 0.5, 1, 5, 10],
        cv=5,
        max_iter=10000,
        random_state=42
    ))
])

enet_cv.fit(X_train, y_train)

best_alpha = enet_cv.named_steps["model"].alpha_
best_l1_ratio = enet_cv.named_steps["model"].l1_ratio_

print("Best alpha:", best_alpha)
print("Best l1_ratio:", best_l1_ratio)


Best alpha: 0.0001
Best l1_ratio: 0.4


In [5]:
# Fit model lần 1 và chọn feature dựa trên coef > threshold = 1e - 8
best_enet = Pipeline([
    ("scaler", StandardScaler()),
    ("model", ElasticNet(
        alpha=best_alpha,
        l1_ratio=best_l1_ratio,
        max_iter=10000,
        random_state=42
    ))
])

best_enet.fit(X_train, y_train)

coef = best_enet.named_steps["model"].coef_
selected_features = X_train.columns[np.abs(coef) > 1e-8].tolist()

print(f"Total selected features: {len(selected_features)}")



Total selected features: 50


In [6]:
# Bộ train với các feature quan trọng
X_train_selected = X_train[selected_features]
X_test_selected  = X_test[selected_features]

# Train lần cuối với feature quan trọng và best params
final_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", ElasticNet(
        alpha=best_alpha,
        l1_ratio=best_l1_ratio,
        max_iter=10000,
        random_state=42
    ))
])

final_model.fit(X_train_selected, y_train)

# Feature importance 
final_coef = final_model.named_steps["model"].coef_

coef_final_df = pd.DataFrame({
    "feature": X_train_selected.columns,
    "coefficient": final_coef
})

# sắp xếp theo |coef| giảm dần 
coef_final_df = coef_final_df.sort_values(
    by="coefficient",
    key=np.abs,
    ascending=False
).reset_index(drop=True)

print(f"[FINAL MODEL] Number of selected features: {len(final_coef)}")

coef_final_df


[FINAL MODEL] Number of selected features: 50


,feature,coefficient
0,original_price,2.277516
1,price,-1.741496
2,review_count,1.084154
3,reputation_score,0.679598
4,price_vs_category,-0.679180
5,discount_amount,-0.429085
6,store_review_count,0.331391
7,category_root_name_Làm Đẹp - Sức Khỏe,0.226006
8,category_root_name_Chăm sóc nhà cửa,0.218039
9,category_root_name_Điện Tử - Điện Lạnh,-0.217934


In [7]:
# Evaluation on test set
y_pred = final_model.predict(X_test_selected)


metrics = {
        "MAE": mean_absolute_error(y_test, y_pred),
        "MSE": mean_squared_error(y_test, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
        "R2": r2_score(y_test, y_pred)
    }

print("\nTest Set Performance:")
for k, v in metrics.items():
        print(f"{k}: {v:.4f}")

results_df = pd.DataFrame({
        "quantity_sold_ground_truth": y_test.values,
        "quantity_sold_predicted": y_pred
    }, index=test_df.index)


results_df.head()

# Save results
predictions_file = "lr_predictions.csv"
results_df.to_csv(predictions_file, index=True)


Test Set Performance:
MAE: 0.5884
MSE: 0.6682
RMSE: 0.8174
R2: 0.8723
